<a href="https://colab.research.google.com/github/Ospino89/spotify-dwh/blob/feature%2Fetl-data-endpoints/notebooks/eda_spotify_ivan_ospino_darcy_escalante.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis Exploratorio de Datos (EDA) — Mi Spotify Wrapped DWH

**Universidad de Pamplona · Bases de Datos II · 2026-I**  
**Profesor:** Juan Alejandro Carrillo Jaimes  
**Integrantes:** Ivan Ospino · Darcy Escalante  
**Fecha:** Mayo 2026

---

> **Nota sobre los datos:** Este análisis refleja el estado del DWH tras las ejecuciones del pipeline ETL realizadas hasta la fecha. Los datos de `fact_listening_history` corresponden a reproducciones reales capturadas desde la Spotify Web API usando el endpoint `GET /v1/me/player/recently-played`. El campo `genres` de `dim_artists` es un array nativo de PostgreSQL (`TEXT[]`) cargado directamente desde el endpoint `GET /v1/me/top/artists`. Los conteos y rankings aumentarán con cada nueva ejecución del ETL.

## 0. Configuración e Imports

In [ ]:
from dotenv import load_dotenv
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from sqlalchemy import create_engine, text
import warnings

warnings.filterwarnings('ignore')

# Cargar variables de entorno — sin credenciales en el código
load_dotenv()  # busca .env en la raíz del proyecto

DATABASE_URL = os.getenv('DATABASE_URL')
assert DATABASE_URL, 'DATABASE_URL no encontrada en .env'

if not DATABASE_URL.startswith('postgresql+psycopg2'):
    DATABASE_URL = DATABASE_URL.replace('postgresql://', 'postgresql+psycopg2://')

engine = create_engine(DATABASE_URL)
print('✅ Conexión exitosa al DWH en Neon.')

def read_sql(query):
    with engine.connect() as conn:
        res = conn.execute(text(query))
        return pd.DataFrame(res.fetchall(), columns=list(res.keys()))

# ── Paleta tema oscuro Spotify ──────────────────────────────────────────────
C_GREEN  = '#1DB954'
C_GREEN2 = '#1aa34a'
C_GREEN3 = '#158a3e'
C_PURPLE = '#a78bfa'
C_AMBER  = '#f59e0b'
C_BLUE   = '#3b82f6'
C_ROSE   = '#f43f5e'
C_BG     = '#0d0d0d'
C_PANEL  = '#161616'
C_GRID   = '#2a2a2a'

GREEN_CMAP = LinearSegmentedColormap.from_list('sp_green', ['#0a3d1f', C_GREEN, '#1ed760'])

plt.rcParams.update({
    'figure.facecolor' : C_BG,
    'axes.facecolor'   : C_PANEL,
    'axes.edgecolor'   : C_GRID,
    'axes.labelcolor'  : '#e0e0e0',
    'xtick.color'      : '#a0a0a0',
    'ytick.color'      : '#a0a0a0',
    'text.color'       : '#ffffff',
    'grid.color'       : C_GRID,
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.45,
    'figure.dpi'       : 100,
    'axes.titlepad'    : 14,
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
    'legend.framealpha': 0.2,
    'legend.edgecolor' : C_GRID,
})
print('✅ Estilo Spotify Dark configurado.')

---
## Paso 1 — Conexión y revisión de los datos

In [ ]:
df_users   = read_sql('SELECT * FROM dwh.dim_users')
df_artists = read_sql('SELECT * FROM dwh.dim_artists')
df_tracks  = read_sql('SELECT * FROM dwh.dim_tracks')
df_history = read_sql('SELECT * FROM dwh.fact_listening_history')

print('Tabla                    Filas  Columnas')
print('-' * 42)
for name, df in [('dim_users', df_users), ('dim_artists', df_artists),
                  ('dim_tracks', df_tracks), ('fact_listening_history', df_history)]:
    print(f'{name:<25} {df.shape[0]:>5}  {df.shape[1]:>7}')

In [ ]:
for name, df in [('dim_users', df_users), ('dim_artists', df_artists),
                  ('dim_tracks', df_tracks), ('fact_listening_history', df_history)]:
    print(f'\n=== {name} ===')
    print(f'Shape: {df.shape}')
    print('Tipos de datos:')
    print(df.dtypes)
    print('% nulos por columna:')
    print((df.isnull().mean() * 100).round(2))
    display(df.head(3))

In [ ]:
# Visualización de calidad de datos
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Porcentaje de Nulos por Columna — Todas las Tablas', fontsize=14, fontweight='bold', y=1.02)

tables = [('dim_users', df_users), ('dim_artists', df_artists),
           ('dim_tracks', df_tracks), ('fact_listening_history', df_history)]

for ax, (title, df) in zip(axes, tables):
    null_pct = (df.isnull().mean() * 100).sort_values(ascending=True)
    colors   = [C_ROSE if v > 0 else C_GREEN for v in null_pct]
    ax.barh(null_pct.index, null_pct.values, color=colors, edgecolor='none', height=0.65)
    ax.set_xlim(0, 105)
    ax.set_xlabel('% Nulos', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.tick_params(axis='y', labelsize=8)
    ax.spines[['top', 'right', 'bottom', 'left']].set_visible(False)
    for i, (col, val) in enumerate(null_pct.items()):
        if val > 0:
            ax.text(val + 1, i, f'{val:.0f}%', va='center', fontsize=7.5, color=C_ROSE)

plt.tight_layout()
plt.savefig('calidad_datos.png', dpi=150, bbox_inches='tight', facecolor=C_BG)
plt.show()

> **Observaciones sobre la calidad de los datos:**  
> La tabla `fact_listening_history` no presenta valores nulos, lo que confirma que el pipeline ETL cargó correctamente todos los campos requeridos. En `dim_artists` puede haber algunos nulos en `genres` para artistas con poca presencia en la plataforma. La tabla `dim_users` tiene nulos esperados en los campos de tokens de Spotify (`spotify_access_token`, `spotify_refresh_token`, `token_expires_at`) que pertenecen a la capa operacional y no afectan el análisis.

---
## Paso 2 — Estadística descriptiva

In [ ]:
print('=== dim_artists — popularity y followers_count ===')
display(df_artists[['popularity', 'followers_count']].describe().round(2))

**Interpretación `dim_artists`:**  
*(Escribe 2 oraciones propias mirando los números que salgan. Con los datos del dashboard, Ivan tiene artistas como Aitana, Coldplay y Michael Jackson con popularidades altas (96–100), lo que sugiere un perfil predominantemente mainstream. Darcy tiene a Silvestre Dangond, Blessd y Diomedes Díaz liderando, artistas con alta popularidad en el mercado latino colombiano.)*

In [ ]:
print('=== dim_tracks — duration_ms y popularity ===')
df_tracks['duration_min'] = df_tracks['duration_ms'] / 60_000
display(df_tracks[['duration_ms', 'duration_min', 'popularity']].describe().round(2))

**Interpretación `dim_tracks`:**  
*(Escribe 2 oraciones propias. Ejemplo: "La duración promedio de las canciones es aproximadamente X minutos, lo que es consistente con el formato estándar de la música pop/latin. La popularidad promedio refleja que [...]")*

---
## Pregunta 1 — ¿Cuáles son mis 10 artistas más escuchados?

In [ ]:
top10 = read_sql("""
    SELECT a.name            AS artist,
           a.followers_count AS followers,
           COUNT(*)          AS plays
    FROM dwh.fact_listening_history h
    JOIN dwh.dim_artists a ON h.artist_id = a.artist_id
    GROUP BY a.name, a.followers_count
    ORDER BY plays DESC
    LIMIT 10
""")

top20 = read_sql("""
    SELECT a.name            AS artist,
           a.followers_count AS followers,
           COUNT(*)          AS plays
    FROM dwh.fact_listening_history h
    JOIN dwh.dim_artists a ON h.artist_id = a.artist_id
    GROUP BY a.name, a.followers_count
    ORDER BY plays DESC
    LIMIT 20
""")

print(top10.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
fig.patch.set_facecolor(C_BG)

# Gráfico 1: Barras horizontales con degradado
ax1 = axes[0]
n = len(top10)
bar_colors = [GREEN_CMAP(v) for v in np.linspace(0.35, 1.0, n)[::-1]]
bars = ax1.barh(
    top10['artist'][::-1], top10['plays'][::-1],
    color=bar_colors, edgecolor='none', height=0.68,
)
max_plays = top10['plays'].max()
for bar in bars:
    w = bar.get_width()
    ax1.text(w + max_plays * 0.02, bar.get_y() + bar.get_height() / 2,
             f'{int(w)}', va='center', ha='left', fontsize=9.5,
             color='white', fontweight='bold')
ax1.set_xlabel('Reproducciones totales', labelpad=8)
ax1.set_title('Top 10 Artistas Más Escuchados', fontweight='bold')
ax1.set_xlim(0, max_plays * 1.22)
ax1.xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=5))
ax1.grid(axis='x', alpha=0.3)
ax1.spines[['top', 'right', 'left', 'bottom']].set_visible(False)

# Gráfico 2: Bubble chart — seguidores globales vs escuchas personales
ax2 = axes[1]
followers_clean = top20['followers'].fillna(0).clip(0)
log_f  = np.log1p(followers_clean)
sizes  = (log_f / (log_f.max() + 1e-9)) * 800 + 80
sc = ax2.scatter(
    followers_clean, top20['plays'],
    s=sizes, c=top20['plays'],
    cmap=GREEN_CMAP, edgecolors='white', linewidths=0.6, alpha=0.88,
)
for _, row in top20.iterrows():
    ax2.annotate(
        row['artist'],
        (row['followers'] if pd.notna(row['followers']) else 0, row['plays']),
        xytext=(7, 4), textcoords='offset points',
        fontsize=7.5, color='#cccccc'
    )
ax2.set_xlabel('Seguidores en Spotify (alcance global)', labelpad=8)
ax2.set_ylabel('Reproducciones en mi historial', labelpad=8)
ax2.set_title('Alcance Global vs Escuchas Personales', fontweight='bold')
ax2.grid(alpha=0.25)
ax2.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
cbar = plt.colorbar(sc, ax=ax2, pad=0.02)
cbar.set_label('Reproducciones', fontsize=9)

plt.suptitle('Análisis de Artistas', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('p1_artistas.png', dpi=150, bbox_inches='tight', facecolor=C_BG)
plt.show()
print(f'Artista #1: {top10.iloc[0]["artist"]} con {top10.iloc[0]["plays"]} reproducciones')

**Interpretación — Pregunta 1:**

**Ivan:** El artista más escuchado en mi historial es Aitana, lo que sí esperaba ya que es un artista que escucho constantemente. Me llamó la atención ver a Dread Mar I y Hola Beats en el top 10, artistas de reggae y electrónica que no recordaba escuchar con tanta frecuencia. En el bubble chart se nota que Coldplay y Michael Jackson tienen millones de seguidores globales pero no son los más repetidos en mi historial personal, lo que muestra que fama global no siempre se traduce en mis escuchas.

**Darcy:** Encabeza Silvestre Dangond con popularidad 100, resultado completamente esperado ya que es el artista que más escucho en el día a día. La presencia de Diomedes Díaz en el top 3 muestra una fuerte raíz en el vallenato clásico, mientras que la aparición de KAROL G y Maluma refleja que también hay espacio para el reggaeton y el latin pop en mi historial.

---
## Pregunta 2 — ¿A qué hora del día escucho más música?

In [ ]:
hourly = read_sql("""
    SELECT hour_of_day, COUNT(*) AS plays
    FROM dwh.fact_listening_history
    GROUP BY hour_of_day ORDER BY hour_of_day
""")
all_hours = pd.DataFrame({'hour_of_day': range(24)})
hourly = all_hours.merge(hourly, on='hour_of_day', how='left').fillna(0)
hourly['plays'] = hourly['plays'].astype(int)
peak_hour = int(hourly.loc[hourly['plays'].idxmax(), 'hour_of_day'])

DAY_ORDER = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
DAY_ES    = {'Monday':'Lunes','Tuesday':'Martes','Wednesday':'Miércoles',
             'Thursday':'Jueves','Friday':'Viernes','Saturday':'Sábado','Sunday':'Domingo'}

daily = read_sql("""
    SELECT day_of_week, COUNT(*) AS plays
    FROM dwh.fact_listening_history GROUP BY day_of_week
""")
daily['day_cat'] = pd.Categorical(daily['day_of_week'], categories=DAY_ORDER, ordered=True)
daily = daily.sort_values('day_cat').reset_index(drop=True)
daily['nombre'] = daily['day_of_week'].map(DAY_ES)

heat_raw = read_sql("""
    SELECT day_of_week, hour_of_day, COUNT(*) AS plays
    FROM dwh.fact_listening_history GROUP BY day_of_week, hour_of_day
""")
heat_pivot = heat_raw.pivot_table(
    index='day_of_week', columns='hour_of_day', values='plays', fill_value=0
).reindex([d for d in DAY_ORDER if d in heat_raw['day_of_week'].unique()]).fillna(0)
for h in range(24):
    if h not in heat_pivot.columns:
        heat_pivot[h] = 0
heat_pivot = heat_pivot[sorted(heat_pivot.columns)].astype(int)
heat_pivot.index = [DAY_ES[d] for d in heat_pivot.index]

print(f'Hora pico: {peak_hour:02d}:00 — {hourly["plays"].max()} reproducciones')

In [ ]:
fig = plt.figure(figsize=(19, 12))
fig.patch.set_facecolor(C_BG)
gs = GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

# Gráfico 1: Reloj polar
ax_polar = fig.add_subplot(gs[0, 0], projection='polar')
ax_polar.set_facecolor('#111111')
theta  = np.linspace(0, 2 * np.pi, 24, endpoint=False)
radii  = hourly['plays'].values.astype(float)
width  = 2 * np.pi / 24
bar_c  = [C_GREEN if i == peak_hour else C_GREEN3 for i in range(24)]
ax_polar.bar(theta, radii, width=width * 0.88,
             color=bar_c, bottom=max(radii.max() * 0.05, 0.1),
             edgecolor=C_BG, linewidth=0.5, alpha=0.92)
ax_polar.set_theta_zero_location('N')
ax_polar.set_theta_direction(-1)
ax_polar.set_xticks(theta)
ax_polar.set_xticklabels([f'{h:02d}h' for h in range(24)], fontsize=7, color='#bbbbbb')
ax_polar.yaxis.set_tick_params(labelcolor='#555555', labelsize=7)
ax_polar.spines['polar'].set_color('#333333')
ax_polar.set_title(f'Distribución Horaria\n(hora pico: {peak_hour:02d}:00)',
                   fontsize=12, fontweight='bold', pad=22)

# Gráfico 2: Barras por día de la semana
ax_day = fig.add_subplot(gs[0, 1])
peak_day_idx = daily['plays'].idxmax()
day_colors = []
for i in daily.index:
    if i == peak_day_idx:
        day_colors.append(C_GREEN)
    elif daily.loc[i, 'day_of_week'] in ('Saturday', 'Sunday'):
        day_colors.append(C_GREEN2)
    else:
        day_colors.append(C_GREEN3)

bars_d = ax_day.bar(daily['nombre'], daily['plays'],
                    color=day_colors, edgecolor='none', width=0.65)
for bar in bars_d:
    ax_day.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + daily['plays'].max() * 0.02,
                str(int(bar.get_height())),
                ha='center', va='bottom', fontsize=10, color='white', fontweight='bold')
ax_day.set_ylabel('Reproducciones')
ax_day.set_title('Reproducciones por Día de la Semana', fontweight='bold')
ax_day.tick_params(axis='x', rotation=28)
ax_day.set_ylim(0, daily['plays'].max() * 1.18)
ax_day.grid(axis='y', alpha=0.3)
ax_day.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
we_patch = mpatches.Patch(color=C_GREEN2, label='Fin de semana')
pk_patch = mpatches.Patch(color=C_GREEN,  label='Día pico')
ax_day.legend(handles=[pk_patch, we_patch], fontsize=9)

# Gráfico 3: Heatmap hora × día
ax_heat = fig.add_subplot(gs[1, :])
sns.heatmap(
    heat_pivot, ax=ax_heat, cmap=GREEN_CMAP,
    linewidths=0.4, linecolor=C_BG,
    annot=True, fmt='d', annot_kws={'size': 7.5},
    cbar_kws={'label': 'Reproducciones', 'shrink': 0.6},
)
ax_heat.set_xlabel('Hora del Día', labelpad=8)
ax_heat.set_ylabel('')
ax_heat.set_title('Mapa de Calor — Reproducciones por Día y Hora', fontweight='bold')
ax_heat.set_xticklabels([f'{h:02d}h' for h in range(24)], fontsize=7.5, rotation=0)
ax_heat.tick_params(axis='y', rotation=0, labelsize=9)

plt.suptitle('¿Cuándo Escuchamos Música?', fontsize=16, fontweight='bold', y=1.01)
plt.savefig('p2_tiempo.png', dpi=150, bbox_inches='tight', facecolor=C_BG)
plt.show()

print(f'Hora pico : {peak_hour:02d}:00 — {hourly["plays"].max()} reproducciones')
print(f'Día pico  : {daily.loc[peak_day_idx, "nombre"]} — {daily.loc[peak_day_idx, "plays"]} reproducciones')

**Interpretación — Pregunta 2:**

**Ivan:** Mi hora pico es las 21:00–22:00, lo que asocio claramente con el tiempo de descanso nocturno después de las actividades del día. El heatmap muestra que hay escucha distribuida en varias horas del día pero con una concentración marcada en la noche, un patrón consistente durante toda la semana.

**Darcy:** La hora pico es las 22:00–23:00, muy similar al patrón de Ivan pero ligeramente más tarde en la noche. Esto sugiere que escucho música principalmente antes de dormir o en las horas de relajación nocturna. Me sorprendió que haya también actividad entre las 05:00 y 07:00, horas que asocio con el traslado en la mañana.

---
## Pregunta 3 — ¿Qué tan popular es la música que escucho?

In [ ]:
pop_df = df_tracks[df_tracks['popularity'].notna() & (df_tracks['popularity'] > 0)][['popularity']].copy()

print('=== Estadísticas de popularidad de top tracks ===')
print(f'Registros con popularidad: {len(pop_df)} de {len(df_tracks)}')
print(pop_df['popularity'].describe().round(2))

mean_pop   = pop_df['popularity'].mean()
median_pop = pop_df['popularity'].median()
print(f'\nMedia:   {mean_pop:.1f}')
print(f'Mediana: {median_pop:.1f}')
print(f'Mínimo:  {pop_df["popularity"].min()}')
print(f'Máximo:  {pop_df["popularity"].max()}')

def clasificar(p):
    if p < 30: return 'underground'
    if p < 60: return 'emerging'
    if p < 80: return 'mainstream'
    return 'viral'

df_tracks['categoria'] = df_tracks['popularity'].apply(
    lambda p: clasificar(p) if pd.notna(p) else 'sin dato'
)
conteo_cat = df_tracks['categoria'].value_counts().reindex(
    ['underground', 'emerging', 'mainstream', 'viral', 'sin dato'], fill_value=0
)
print('\nCanciones por categoría:')
print(conteo_cat)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor(C_BG)

# Histograma + KDE
ax1 = axes[0]
if len(pop_df) > 1:
    ax1.hist(pop_df['popularity'], bins=20, color=C_GREEN, edgecolor=C_BG,
             alpha=0.65, density=True, label='Histograma')
    sns.kdeplot(data=pop_df['popularity'], ax=ax1, color=C_AMBER,
                linewidth=2.5, label='Densidad (KDE)', fill=True, alpha=0.12)
    ax1.axvline(mean_pop,   color='white',  linestyle='--', linewidth=1.5,
                label=f'Media: {mean_pop:.1f}')
    ax1.axvline(median_pop, color=C_PURPLE, linestyle=':',  linewidth=1.5,
                label=f'Mediana: {median_pop:.1f}')
ax1.set_xlabel('Popularidad (0 = underground, 100 = viral)', labelpad=8)
ax1.set_ylabel('Densidad', labelpad=8)
ax1.set_title('Distribución de Popularidad (top tracks)', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)
ax1.spines[['top', 'right', 'left', 'bottom']].set_visible(False)

# Barras por categoría
ax2 = axes[1]
cats_to_show = ['underground', 'emerging', 'mainstream', 'viral']
cat_colors   = {'underground': '#535353', 'emerging': '#b3b3b3',
                'mainstream': C_GREEN, 'viral': '#FFD700'}
vals = [conteo_cat[c] for c in cats_to_show]
bars = ax2.bar(cats_to_show, vals,
               color=[cat_colors[c] for c in cats_to_show], edgecolor='none')
max_val = max(vals) if max(vals) > 0 else 1
for bar in bars:
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + max_val * 0.02,
             str(int(bar.get_height())),
             ha='center', fontsize=12, fontweight='bold', color='white')
ax2.set_title('Canciones por Categoría de Popularidad', fontweight='bold')
ax2.set_xlabel('Categoría', labelpad=8)
ax2.set_ylabel('Cantidad de canciones', labelpad=8)
ax2.grid(axis='y', alpha=0.3)
ax2.spines[['top', 'right', 'left', 'bottom']].set_visible(False)

plt.suptitle('¿Qué Tan Popular Es La Música Que Escuchamos?', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('p3_popularidad.png', dpi=150, bbox_inches='tight', facecolor=C_BG)
plt.show()

**Interpretación — Pregunta 3:**

**Ivan:** El top artistas de mi cuenta muestra popularidades entre 92 y 100, lo que indica que soy un oyente claramente mainstream. Las canciones de Aitana, Coldplay y Michael Jackson son artistas con altísima exposición global, así que no me sorprende que la popularidad promedio sea elevada. Me resultaría interesante ver si las canciones que escucho de Dread Mar I o artistas de reggae local bajan ese promedio hacia la zona emerging.

**Darcy:** Con artistas como Silvestre Dangond (pop 100), Blessd (pop 98) y Diomedes Díaz (pop 96), mi perfil es también muy mainstream dentro del contexto de la música latina colombiana. La presencia de Diomedes Díaz, un artista clásico del vallenato, con popularidad tan alta muestra que ese género tiene un alcance enorme en la plataforma.

---
## Pregunta 4 — ¿Qué géneros dominan mi historial?

In [ ]:
# Método SQL con UNNEST (como exige la rúbrica)
genres_sql = read_sql("""
    SELECT LOWER(TRIM(g)) AS genre,
           COUNT(*)       AS plays
    FROM dwh.fact_listening_history h
    JOIN dwh.dim_artists a ON h.artist_id = a.artist_id
    CROSS JOIN UNNEST(a.genres) AS g
    WHERE TRIM(g) <> ''
    GROUP BY LOWER(TRIM(g))
    ORDER BY plays DESC
    LIMIT 20
""")

print('=== Top 20 géneros (via SQL UNNEST) ===')
print(genres_sql.to_string(index=False))

In [ ]:
# Equivalente Pandas nativo con .explode()
merged = df_history.merge(df_artists[['artist_id', 'genres']], on='artist_id', how='left')
merged['genres'] = merged['genres'].apply(
    lambda g: [x.strip().lower() for x in g if isinstance(x, str) and x.strip()]
    if isinstance(g, list) else []
)
all_genres = merged['genres'].explode().dropna()
all_genres = all_genres[all_genres.str.strip() != '']
genres_df  = all_genres.value_counts().reset_index()
genres_df.columns = ['genre', 'plays']

n_total = len(genres_df)
top15_g = genres_df.head(15)
top8_g  = genres_df.head(8)
print(f'Géneros únicos detectados: {n_total}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
fig.patch.set_facecolor(C_BG)

# Barras horizontales top 15
ax1 = axes[0]
n_g = len(top15_g)
g_colors = [GREEN_CMAP(v) for v in np.linspace(0.3, 1.0, n_g)[::-1]]
bars_g = ax1.barh(
    top15_g['genre'][::-1], top15_g['plays'][::-1],
    color=g_colors, edgecolor='none', height=0.72,
)
max_g = top15_g['plays'].max()
for bar in bars_g:
    ax1.text(bar.get_width() + max_g * 0.02,
             bar.get_y() + bar.get_height() / 2,
             str(int(bar.get_width())),
             va='center', ha='left', fontsize=9.5, color='white', fontweight='bold')
ax1.set_xlabel('Apariciones en el historial', labelpad=8)
ax1.set_title('Top 15 Géneros Más Escuchados', fontweight='bold')
ax1.set_xlim(0, max_g * 1.22)
ax1.grid(axis='x', alpha=0.3)
ax1.spines[['top', 'right', 'left', 'bottom']].set_visible(False)

# Donut chart top 8 + Otros
ax2 = axes[1]
otros   = genres_df.iloc[8:]['plays'].sum() if len(genres_df) > 8 else 0
donut_v = list(top8_g['plays']) + ([otros] if otros > 0 else [])
donut_l = list(top8_g['genre']) + ([f'Otros ({n_total - 8})'] if otros > 0 else [])
donut_c = [C_GREEN, C_GREEN2, C_GREEN3, '#0f6630',
           C_PURPLE, C_BLUE, C_AMBER, C_ROSE, '#666666'][:len(donut_v)]
wedges, texts, pcts = ax2.pie(
    donut_v, labels=donut_l, colors=donut_c,
    autopct='%1.1f%%', pctdistance=0.80, startangle=90,
    wedgeprops=dict(width=0.52, edgecolor=C_BG, linewidth=2.5),
)
for t in texts: t.set(fontsize=9,  color='white')
for p in pcts:  p.set(fontsize=8,  color='white', fontweight='bold')
ax2.text(0, 0, f'{n_total}\ngéneros',
         ha='center', va='center', fontsize=12, fontweight='bold',
         color='white', linespacing=1.6)
ax2.set_title('Proporción de Géneros (Top 8 + Otros)', fontweight='bold')

plt.suptitle('Análisis de Géneros Musicales', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('p4_generos.png', dpi=150, bbox_inches='tight', facecolor=C_BG)
plt.show()

In [ ]:
# Curva de concentración de géneros
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(C_BG)

cumsum_pct = genres_df['plays'].cumsum() / genres_df['plays'].sum() * 100
ax.plot(range(1, len(genres_df) + 1), cumsum_pct,
        color=C_GREEN, linewidth=2.5, zorder=3)
ax.fill_between(range(1, len(genres_df) + 1), cumsum_pct,
                alpha=0.15, color=C_GREEN)

idx_80 = int((cumsum_pct >= 80).idxmax()) + 1
ax.axhline(80, color=C_AMBER, linestyle='--', linewidth=1.3, alpha=0.8)
ax.axvline(idx_80, color=C_AMBER, linestyle='--', linewidth=1.3, alpha=0.8)
ax.annotate(
    f'{idx_80} géneros\nexplican el 80%\nde las escuchas',
    xy=(idx_80, 80), xytext=(idx_80 + 2, 55),
    arrowprops=dict(arrowstyle='->', color=C_AMBER, lw=1.5),
    fontsize=9.5, color=C_AMBER, fontweight='bold'
)
ax.set_xlabel('Número de géneros (ordenados por frecuencia)', labelpad=8)
ax.set_ylabel('% acumulado de reproducciones', labelpad=8)
ax.set_title('Curva de Concentración de Géneros', fontweight='bold')
ax.set_ylim(0, 102)
ax.grid(alpha=0.3)
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
plt.tight_layout()
plt.savefig('p4_concentracion.png', dpi=150, bbox_inches='tight', facecolor=C_BG)
plt.show()
print(f'Con {idx_80} géneros se cubre el 80% de todas las reproducciones.')

**Interpretación — Pregunta 4:**

**Ivan:** El género dominante en mi historial es `latin` con 209 plays, seguido de `latin pop` con 130, lo que refleja mi afinidad por la música en español. Me sorprendió ver `vallenato` tan alto en el top 3, un género que quizás no asociaba tan fuertemente conmigo hasta ver los datos. La curva de concentración muestra que probablemente solo 3 o 4 géneros explican la mayoría de mis escuchas, lo que indica gustos bastante definidos y concentrados.

**Darcy:** El género `latin` también domina mi perfil, pero lo interesante es la diversidad: junto al vallenato y el latin pop aparecen con fuerza `alternative rock`, `nu metal` y `heavy metal`, géneros que contrastan fuertemente con el lado latino. Esto refleja que tengo gustos muy variados que van del vallenato tradicional al metal, algo que definitivamente no esperaba ver reflejado tan claramente en los datos.

---
## Paso 4 — Conclusiones individuales

### 💬 Ivan Ospino

**¿Qué aprendí de mis propios datos que no sabía antes?**  
Sabía que escuchaba a Aitana seguido, pero no imaginaba que fuera tan claramente el artista #1 en mi historial. Lo que más me sorprendió fue ver a Dread Mar I y artistas de reggae en mi top 10, géneros que escucho pero que no percibía como tan frecuentes. Los datos también confirmaron algo que sospechaba: escucho música principalmente de noche, entre las 21:00 y las 23:00, horas que asocio con tiempo personal después de estudiar o trabajar.

**¿Qué pregunta quise hacerle a los datos pero el modelo actual no me permitió responder?**  
Quise saber si las canciones que escucho en la noche tienen características de audio distintas a las que escucho en la mañana — por ejemplo, si son más tranquilas o con menos energía. Eso requeriría tener los audio features de Spotify (`energy`, `valence`, `tempo`) en el DWH. Para responderla le agregaría a `dim_tracks` columnas como `energy`, `valence`, `danceability` y `tempo`, que están disponibles en el endpoint `GET /v1/audio-features`.

### 💬 Darcy Escalante

**¿Qué aprendí de mis propios datos que no sabía antes?**  
No esperaba ver géneros como `nu metal`, `alternative rock` y `heavy metal` tan arriba en mi top de géneros. Siempre me identifico como oyente de música latina, pero los datos muestran que hay una doble personalidad musical: vallenato y salsa de un lado, metal y rock alternativo del otro. También descubrí que mi hora pico (22:00–23:00) es más tardía de lo que pensaba — no me daba cuenta de que escuchaba tanto antes de dormir.

**¿Qué pregunta quise hacerle a los datos pero el modelo actual no me permitió responder?**  
Quise analizar si hay diferencia entre los géneros que escucho entre semana versus los fines de semana — mi hipótesis es que entre semana escucho más vallenato y los fines de semana más metal. El modelo actual sí tiene `day_of_week` en `fact_listening_history`, pero para llegar al género necesito hacer un JOIN con `dim_artists` y hacer el `UNNEST` del array, lo cual es posible. Lo que faltaría sería una tabla `dim_genres` normalizada con una relación many-to-many hacia `dim_artists`, para poder filtrar y agrupar por género de forma más eficiente sin tener que desanidar en cada consulta.